In [1]:
import numpy as np
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from scipy.stats import norm

# =========================================================
# WEEK 11 — FUNCTION 7 (CLUSTER-AWARE BO, STRICT [0,1] BOUNDS)
# - Clustering lens (KMeans + silhouette) to find promising regions
# - Candidate mix:
#     (1) Sobol global exploration (space-filling)
#     (2) Trust region around best point (exploitation)
#     (3) Local around best-performing cluster centroid (centroid trend)
#     (4) Boundary probes between top cluster and others (boundary tightening)
# - Min-distance filter to avoid duplicates
# - Mix EI (exploit) + UCB (explore), transparent logs
# - Prints x_next with 6 decimals
# =========================================================

# -----------------------------
# 1) Data (Function 7)
# -----------------------------
X_raw = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654 ],
    [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143, 0.58523563],
    [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164 ],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 , 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825 ],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848, 0.6555429 ],
    [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 , 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 , 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 , 0.07966402],
    [1.04245 , 1.024693, 1.02457 , 1.061017, 1.098654, 1.051013],
    [0.019976, 0.432955, 0.301662, 0.169496, 0.348651, 0.743371],
    [0.611853, 0.139495, 0.292145, 0.366362, 0.45607 , 0.785175],
    [0.015006, 0.390905, 0.178469, 0.119929, 0.088415, 0.904408],
    [0.046821, 0.309546, 0.608802, 0.064364, 0.39334 , 0.990644],
    [0.011478, 0.62027 , 0.525606, 0.053535, 0.52488 , 0.666127],
    [0.028679, 0.235471, 0.148723, 0.076614, 0.11285 , 0.837107],
    [0.069198, 0.39455 , 0.352452, 0.093928, 0.370707, 0.725655],
    [0.000000, 0.367783, 0.346341, 0.052998, 0.363011, 0.730958],
    [0.000000, 0.319509, 0.283024, 0.207785, 0.339392, 0.737674]
])

y_raw = np.array([
    6.04432696e-01, 5.62753067e-01, 7.50323668e-03, 6.14243025e-02,
    2.73046801e-01, 8.37465723e-02, 1.36496830e+00, 9.26449549e-02,
    1.78695987e-02, 3.35649360e-02, 7.35163042e-02, 2.06309698e-01,
    8.82563400e-03, 2.68400317e-01, 6.11525528e-01, 1.47981826e-02,
    2.74892508e-01, 6.67632469e-02, 4.21183545e-02, 2.70146502e-03,
    1.82090730e-02, 7.01602756e-03, 1.00506611e-01, 4.75395516e-01,
    6.75141631e-01, 5.16457219e-01, 3.77747962e-03, 3.13433331e-03,
    2.13425228e-02, 9.54111589e-02, 4.636858051500375e-06, 1.680828424430851,
    1.1170576710554418, 0.44099891630237703, 0.8950628737420184, 0.664856997347448,
    0.6583383225997628, 1.6173276124769211, 1.3305832886811908, 2.1047631312317456
])

# -----------------------------
# 2) Config (Week 11)
# -----------------------------
RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)

# Candidate search budget (rebalanced toward cluster-local + boundary probing)
N_GLOBAL = 3500
N_LOCAL_BEST = 2200
N_LOCAL_CENTROID = 1800
N_BOUNDARY = 1200

# Trust-region scales
LOCAL_SCALE_BEST = 0.045      # tighter than Week 10: exploit around incumbent
LOCAL_SCALE_CENT = 0.060      # slightly looser around centroid
BOUNDARY_JITTER = 0.025       # jitter around boundary midpoints

MIN_DIST = 7.5e-4             # a bit stricter to avoid redundant queries

# Acquisition knobs (slightly more exploit now)
EI_XI = 0.008
UCB_BETA = 1.6
MIX_ALPHA = 0.75

# Clustering knobs
K_RANGE = range(2, 7)         # small dataset -> keep k modest
KMEANS_N_INIT = 20

# -----------------------------
# 3) Acquisition helpers
# -----------------------------
def gaussian_ei(mu, sigma, f_best, xi=0.0):
    sigma = np.maximum(sigma, 1e-12)
    z = (mu - f_best - xi) / sigma
    return (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)

def gaussian_ucb(mu, sigma, beta=1.0):
    return mu + beta * np.maximum(sigma, 1e-12)

def zscore(a):
    s = np.std(a)
    return (a - np.mean(a)) / s if s > 1e-12 else (a - np.mean(a))

# -----------------------------
# 4) Candidate generation
# -----------------------------
def sobol_global_candidates(n, d, seed):
    eng = torch.quasirandom.SobolEngine(dimension=d, scramble=True, seed=seed)
    return eng.draw(n).numpy()

def local_gaussian_candidates(center, n, scale, seed):
    rng = np.random.RandomState(seed)
    Xl = center + rng.normal(0.0, scale, size=(n, center.size))
    return np.clip(Xl, 0.0, 1.0)

def boundary_candidates(centroid_top, centroids_other, n, jitter, seed):
    """
    Create candidates near the midpoints between top centroid and other centroids,
    then jitter them (boundary tightening).
    """
    rng = np.random.RandomState(seed)
    if centroids_other.shape[0] == 0:
        return np.empty((0, centroid_top.size))

    mids = 0.5 * (centroid_top[None, :] + centroids_other)
    reps = int(np.ceil(n / mids.shape[0]))
    base = np.vstack([mids for _ in range(reps)])[:n]
    Xb = base + rng.normal(0.0, jitter, size=base.shape)
    return np.clip(Xb, 0.0, 1.0)

def min_dist_filter(Xcand, Xtrain, thr):
    if Xcand.shape[0] == 0:
        return Xcand
    thr2 = thr * thr
    dist2 = ((Xcand[:, None, :] - Xtrain[None, :, :]) ** 2).sum(axis=2)
    keep = dist2.min(axis=1) > thr2
    return Xcand[keep]

# -----------------------------
# 5) Clustering helper (KMeans + silhouette)
# -----------------------------
def pick_kmeans_and_clusters(X01, y, seed):
    """
    Cluster in standardized space. Choose k via silhouette (simple, robust for small n).
    Returns: best_k, labels, centroids in original [0,1] space, silhouette
    """
    xs = StandardScaler()
    Xs = xs.fit_transform(X01)

    best = None
    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=seed, n_init=KMEANS_N_INIT)
        labels = km.fit_predict(Xs)
        # silhouette requires >=2 clusters and no empty cluster; KMeans ensures that
        sil = silhouette_score(Xs, labels)
        if (best is None) or (sil > best["sil"]):
            centroids01 = xs.inverse_transform(km.cluster_centers_)
            best = {"k": k, "labels": labels, "centroids01": np.clip(centroids01, 0.0, 1.0), "sil": sil}
    return best["k"], best["labels"], best["centroids01"], best["sil"]

# -----------------------------
# 6) Main
# -----------------------------
def main():
    # Strict bounds (handles >1 row cleanly)
    X = np.clip(X_raw, 0.0, 1.0)
    d = X.shape[1]

    # Best observed
    best_idx = int(np.argmax(y_raw))
    x_best = X[best_idx].copy()
    f_best = float(np.max(y_raw))

    # ---------- Clustering lens ----------
    k, labels, centroids01, sil = pick_kmeans_and_clusters(X, y_raw, RANDOM_STATE)

    # Cluster performance summaries
    cluster_ids = np.unique(labels)
    cluster_stats = []
    for cid in cluster_ids:
        idx = np.where(labels == cid)[0]
        ys = y_raw[idx]
        cluster_stats.append({
            "cid": int(cid),
            "n": int(idx.size),
            "y_mean": float(np.mean(ys)),
            "y_best": float(np.max(ys)),
        })
    cluster_stats = sorted(cluster_stats, key=lambda t: (t["y_best"], t["y_mean"]), reverse=True)
    top_cid = cluster_stats[0]["cid"]
    centroid_top = centroids01[top_cid].copy()

    other_cids = [c for c in cluster_ids if c != top_cid]
    centroids_other = centroids01[other_cids] if len(other_cids) else np.empty((0, d))

    # ---------- GP surrogate ----------
    xs = StandardScaler()
    ys = StandardScaler()
    Xs = xs.fit_transform(X)
    ys_scaled = ys.fit_transform(y_raw.reshape(-1, 1)).ravel()

    kernel = (
        C(1.0, (1e-2, 1e2))
        * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2), nu=2.5)
        + WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-8, 1e-1))
    )
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=False,
        random_state=RANDOM_STATE,
        n_restarts_optimizer=0
    )
    gp.fit(Xs, ys_scaled)

    # ---------- Candidate mix ----------
    Xg = sobol_global_candidates(N_GLOBAL, d, RANDOM_STATE)

    # Local around current best (tight trust region)
    Xl_best = local_gaussian_candidates(x_best, N_LOCAL_BEST, LOCAL_SCALE_BEST, RANDOM_STATE + 1)

    # Local around top cluster centroid (centroid trend)
    Xl_cent = local_gaussian_candidates(centroid_top, N_LOCAL_CENTROID, LOCAL_SCALE_CENT, RANDOM_STATE + 2)

    # Boundary probes (between top cluster and others)
    Xb = boundary_candidates(centroid_top, centroids_other, N_BOUNDARY, BOUNDARY_JITTER, RANDOM_STATE + 3)

    Xcand = np.vstack([Xg, Xl_best, Xl_cent, Xb])
    Xcand = min_dist_filter(Xcand, X, MIN_DIST)

    # Predict mean/std in original y space
    mu_s, std_s = gp.predict(xs.transform(Xcand), return_std=True)
    mu = ys.inverse_transform(mu_s.reshape(-1, 1)).ravel()
    sigma = np.maximum(std_s * float(ys.scale_[0]), 1e-12)

    # Acquisition
    ei = gaussian_ei(mu, sigma, f_best, xi=EI_XI)
    ucb = gaussian_ucb(mu, sigma, beta=UCB_BETA)
    score = MIX_ALPHA * zscore(ei) + (1.0 - MIX_ALPHA) * zscore(ucb)

    best_cand = int(np.argmax(score))
    x_next = np.round(np.clip(Xcand[best_cand], 0.0, 1.0), 6)

    # ---------- Transparent logs ----------
    print("CURRENT BEST OBSERVED")
    print("f_best =", f_best)
    print("x_best =", np.round(x_best, 6))

    print("\nCLUSTERING LENS (KMeans)")
    print(f"chosen_k = {k} | silhouette = {sil:.4f}")
    print("cluster ranking (by y_best then y_mean):")
    for t in cluster_stats:
        mark = " <== top cluster" if t["cid"] == top_cid else ""
        print(f"  cid={t['cid']} | n={t['n']} | y_best={t['y_best']:.6f} | y_mean={t['y_mean']:.6f}{mark}")
    print("top cluster centroid =", np.round(centroid_top, 6))

    print("\nSURROGATE (GP) KERNEL")
    print(gp.kernel_)
    if hasattr(gp.kernel_, "k1") and hasattr(gp.kernel_.k1, "k2"):
        ls = np.array(gp.kernel_.k1.k2.length_scale, dtype=float).ravel()
        print("lengthscales (smaller => more sensitive):", np.round(ls, 4))

    print("\nCANDIDATE MIX (after min-dist filter)")
    print("N_global =", N_GLOBAL, "| N_local_best =", N_LOCAL_BEST,
          "| N_local_centroid =", N_LOCAL_CENTROID, "| N_boundary =", N_BOUNDARY)
    print("Xcand_kept =", int(Xcand.shape[0]))

    print("\nACQUISITION BREAKDOWN @ x_next")
    print("mu =", float(mu[best_cand]))
    print("sigma =", float(sigma[best_cand]))
    print("EI =", float(ei[best_cand]))
    print("UCB =", float(ucb[best_cand]))

    np.set_printoptions(suppress=True, formatter={"float_kind": lambda v: f"{v:.6f}"})
    print("\nRECOMMENDED NEXT POINT")
    print("x_next =", x_next)

if __name__ == "__main__":
    main()


C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than avai

CURRENT BEST OBSERVED
f_best = 2.1047631312317456
x_best = [0.       0.319509 0.283024 0.207785 0.339392 0.737674]

CLUSTERING LENS (KMeans)
chosen_k = 6 | silhouette = 0.2848
cluster ranking (by y_best then y_mean):
  cid=2 | n=10 | y_best=2.104763 | y_mean=1.123312 <== top cluster
  cid=3 | n=10 | y_best=1.117058 | y_mean=0.338463
  cid=1 | n=4 | y_best=0.604433 | y_mean=0.297333
  cid=5 | n=4 | y_best=0.562753 | y_mean=0.225642
  cid=0 | n=7 | y_best=0.100507 | y_mean=0.034461
  cid=4 | n=5 | y_best=0.095411 | y_mean=0.029429
top cluster centroid = [0.042503 0.418708 0.328804 0.152632 0.305944 0.779505]

SURROGATE (GP) KERNEL
0.736**2 * Matern(length_scale=[3.49, 100, 100, 1.16, 0.665, 1.28], nu=2.5) + WhiteKernel(noise_level=0.0559)
lengthscales (smaller => more sensitive): [  3.4945 100.     100.       1.1587   0.6649   1.285 ]

CANDIDATE MIX (after min-dist filter)
N_global = 3500 | N_local_best = 2200 | N_local_centroid = 1800 | N_boundary = 1200
Xcand_kept = 8700

ACQUISITION B

C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
